# Notebook 03 — Stockiness Measures (Revised)

**Project:** Scènes à Faire Computational Framework (v4 rebuild)  
**Purpose:** Compute six stockiness measures grounded in three intellectual traditions: information theory, corpus linguistics, and Bayesian inference.

**Changes from v1 of this notebook:**
1. **Universality filter recalibrated.** Previously flagged elements appearing in ≥14 of 17 genres as universal, which hid `sheriff`, `horse`, `gun`, etc. Now uses a stricter 17/17 cutoff AND switches from a binary flag to a continuous breadth penalty that downweights without hiding.
2. **Minimum film filter added.** Elements appearing in fewer than 5 distinct films are dropped. This kills the hapax-legomena problem (proper nouns like `rocklin` that dominated PMI's top lists for Western).
3. **Dunning's log-likelihood ratio (G²) added.** The corpus-linguistics gold standard for keyness, statistically principled in a way PMI is not. Used by every major corpus tool (AntConc, Sketch Engine, BYU Law-Corpus).
4. **Posterior Stockiness P(g|e) added.** Directly answers the Bayesian question scènes à faire actually asks: when a court sees element X, what's the probability the work is genre Y?
5. **Ensemble redesigned.** Now combines four robust measures (PS, TF-IDF, G², Posterior) and excludes raw PMI, whose vulnerability to rare events made the prior ensemble unreliable.
6. **Instance filter de-duplicated (October 2026).** The 10-instance threshold now uses `raw_instance_count`, which counts each instance once per film, instead of the post-explode sum `total_instance_count`, which counted a film's instances once per genre tag. `total_instance_count` is still stored and still feeds TF, PMI, G², and entropy.

**Six measures computed for every (element, genre) pair:**
- **Primary Stockiness (PS)** — film-based frequency × distinctiveness
- **TF-IDF** — instance-based term frequency × log inverse genre-frequency
- **PMI** — pointwise mutual information (kept for completeness but flagged as fragile)
- **Dunning's G²** — log-likelihood ratio of element-genre association vs. chance
- **Posterior Stockiness** — P(genre | element), the Bayesian posterior
- **Ensemble** — z-scored average of the four robust measures (PS, TF-IDF, G², Posterior)

**Runtime:** ~3-5 minutes on default CPU.  
**Inputs:** `element_index.parquet`, `movies_union.parquet`.  
**Outputs:** `stockiness_long.parquet`, `top_stock_by_genre.parquet`, `stockiness_report.txt`.

## 1. Setup

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q pyarrow

In [8]:
import os
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option('display.max_colwidth', 80)

In [9]:
PROJECT_ROOT = '/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational'

UNION_PATH = f'{PROJECT_ROOT}/Analysis/v4_rebuild/01c_union/movies_union.parquet'
INDEX_PATH = f'{PROJECT_ROOT}/Analysis/v4_rebuild/02_elements/element_index.parquet'

OUTPUT_DIR = f'{PROJECT_ROOT}/Analysis/v4_rebuild/03_stockiness'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(UNION_PATH), 'Union parquet missing. Run Notebook 01c.'
assert os.path.exists(INDEX_PATH), 'Element index missing. Run Notebook 02.'

CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western'
]
N_GENRES = len(CANONICAL_GENRES)

# Element must appear in this many films (across all genres) to be retained.
# Kills proper-noun hapax that dominated PMI's top lists in v1.
MIN_FILMS_PER_ELEMENT = 5

# Element must have this many total instances to be retained.
MIN_FREQ_THRESHOLD = 10

# Universal = appears in ALL 17 genres (not 14). Much stricter than v1.
# Elements still get tagged is_universal, but tops are no longer filtered by this flag.
UNIVERSAL_THRESHOLD = 17

# Top-K for displays
TOP_K = 15

## 2. Load and Join

Join the element index with the union corpus on `union_id`, explode multi-label genres so every (element, genre) pair becomes a row. Handle the parquet list→ndarray type quirk explicitly.

In [10]:
print('Loading element index...')
ei = pd.read_parquet(INDEX_PATH)
print(f'  {len(ei):,} element rows across {ei["union_id"].nunique():,} films')

print('\nLoading union corpus...')
union = pd.read_parquet(UNION_PATH)

# Fix list-vs-array type issue from parquet round-trip
union['genres_canonical'] = union['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)

genre_lookup = union[['union_id', 'genres_canonical']].copy()
print(f'  {len(union):,} films')

Loading element index...
  5,658,714 element rows across 45,956 films

Loading union corpus...
  45,958 films


In [11]:
print('Joining elements with genre labels and exploding multi-label genres...')
joined = ei.merge(genre_lookup, on='union_id', how='inner')
joined = joined.explode('genres_canonical').rename(columns={'genres_canonical': 'genre'})
joined = joined[joined['genre'].isin(CANONICAL_GENRES)].reset_index(drop=True)
print(f'  After explode: {len(joined):,} (element × genre) rows')

films_per_genre = (joined[['union_id', 'genre']]
                   .drop_duplicates()
                   .groupby('genre')
                   .size())
print('\nFilms per genre (films with at least one canonical genre label):')
for g in CANONICAL_GENRES:
    print(f'  {g:<18} {films_per_genre.get(g, 0):>6,}')

Joining elements with genre labels and exploding multi-label genres...
  After explode: 13,645,095 (element × genre) rows

Films per genre (films with at least one canonical genre label):
  Action              7,775
  Adventure           6,146
  Animation           2,835
  Comedy             14,722
  Crime               5,564
  Drama              22,364
  Family              4,164
  Fantasy             2,429
  History             2,882
  Horror              4,363
  Music               2,926
  Mystery             2,681
  Romance             8,219
  Science Fiction     2,590
  Thriller            7,447
  War                 1,658
  Western             1,362


## 3. Element × Genre Frequencies, With Filtering

Build the (element, type, genre) frequency table with both instance counts and film counts. Apply two filters: minimum total instances (`≥10`) and minimum total films (`≥5`). The film filter is the key addition — it kills proper-noun hapax (a name appearing 50 times in one Wikipedia plot would have passed the instance filter alone).

The instance threshold counts each instance once per film (`raw_instance_count`, computed from the pre-explode element index). The exploded sum `total_instance_count`, in which a film with k genre tags contributes each instance k times, is still stored because the per-genre measures treat each (film, genre) pair as a unit and the element-genre entropy needs `instance_count / total_instance_count` to be a distribution over genres. Before October 2026 the filter was applied to the exploded sum, which let some elements through on the strength of multi-tagged films; the manuscript and Methods Appendix describe the filter as counting each instance once, and the code now does so.

In [12]:
print('Computing element × genre frequency tables...')

# Instance count per (element, type, genre) cell
eg_count = (joined
            .groupby(['element', 'element_type', 'genre'])
            .size()
            .reset_index(name='instance_count'))

# Film count per (element, type, genre): distinct films in g containing e
eg_films = (joined[['element', 'element_type', 'genre', 'union_id']]
            .drop_duplicates()
            .groupby(['element', 'element_type', 'genre'])
            .size()
            .reset_index(name='film_count'))

freq = eg_count.merge(eg_films, on=['element', 'element_type', 'genre'], how='inner')
print(f'  {len(freq):,} (element, type, genre) cells before filter')

Computing element × genre frequency tables...
  1,958,454 (element, type, genre) cells before filter


In [ ]:
# Marginal aggregates (computed BEFORE filtering — they're properties of the element across all genres)

elem_total = (freq.groupby(['element', 'element_type'])['instance_count']
              .sum()
              .reset_index(name='total_instance_count'))

elem_films_total = (joined[['element', 'element_type', 'union_id']]
                    .drop_duplicates()
                    .groupby(['element', 'element_type'])
                    .size()
                    .reset_index(name='total_film_count'))

# De-duplicated instance count for the instance filter.
# instance_count above is a row count over the post-explode table, so a film
# carrying k genre tags contributes each of its instances k times to
# total_instance_count. That exploded sum is kept as is because the per-genre
# measures treat each (film, genre) pair as a unit and element_entropy needs
# instance_count / total_instance_count to be a distribution over genres.
# The 10-instance filter, however, is meant to count each instance once per
# film (as the manuscript and the Methods Appendix state). raw_instance_count
# is the same per-film row count that the notebook sums into instance_count,
# aggregated over films with each film counted once, restricted to the films
# that survive the inner merge with genre_lookup (the films that feed joined).
# total_film_count is already de-duplicated and is unchanged.
films_in_corpus = genre_lookup['union_id'].unique()
elem_raw_total = (ei[ei['union_id'].isin(films_in_corpus)]
                  .groupby(['element', 'element_type'])
                  .size()
                  .reset_index(name='raw_instance_count'))

elem_genre_span = (freq.groupby(['element', 'element_type'])['genre']
                   .nunique()
                   .reset_index(name='n_genres_present'))

genre_total_instances = freq.groupby('genre')['instance_count'].sum().to_dict()

freq = freq.merge(elem_total, on=['element', 'element_type'])
freq = freq.merge(elem_films_total, on=['element', 'element_type'])
freq = freq.merge(elem_raw_total, on=['element', 'element_type'])
freq = freq.merge(elem_genre_span, on=['element', 'element_type'])

# Apply BOTH filters. The instance threshold uses raw_instance_count (each
# instance counted once per film), not the exploded total_instance_count.
before = len(freq)
n_pairs_before = freq[['element', 'element_type']].drop_duplicates().shape[0]
n_dropped_by_dedup = (
    freq[['element', 'element_type', 'total_instance_count',
          'raw_instance_count', 'total_film_count']]
    .drop_duplicates(subset=['element', 'element_type'])
    .query('total_instance_count >= @MIN_FREQ_THRESHOLD and '
           'raw_instance_count < @MIN_FREQ_THRESHOLD and '
           'total_film_count >= @MIN_FILMS_PER_ELEMENT')
    .shape[0]
)
freq = freq[
    (freq['raw_instance_count'] >= MIN_FREQ_THRESHOLD) &
    (freq['total_film_count'] >= MIN_FILMS_PER_ELEMENT)
].reset_index(drop=True)
after = len(freq)

n_unique_elements_kept = freq[['element', 'element_type']].drop_duplicates().shape[0]
n_dropped_by_film_filter = (
    elem_films_total.merge(elem_raw_total, on=['element', 'element_type'])
    .query('total_film_count < @MIN_FILMS_PER_ELEMENT and raw_instance_count >= @MIN_FREQ_THRESHOLD')
    .shape[0]
)

print(f'After filtering (≥{MIN_FREQ_THRESHOLD} instances, each counted once per film, '
      f'AND ≥{MIN_FILMS_PER_ELEMENT} films):')
print(f'  Cells: {before:,} → {after:,}')
print(f'  Unique elements: {n_pairs_before:,} → {n_unique_elements_kept:,}')
print(f'  Elements killed specifically by the film filter (hapax noise): {n_dropped_by_film_filter:,}')
print(f'  Elements that pass the film filter and the exploded-sum instance count '
      f'but fail the de-duplicated instance count: {n_dropped_by_dedup:,}')

## 4. Universality Tag (Continuous and Binary)

Compute breadth in two forms:
- `is_universal_strict` (binary): appears in all 17 genres. This is the strict definition.
- `breadth_penalty` (continuous): n_genres_present / 17. Used for the breadth-adjusted ranking views so genuinely universal narrative words (`man`, `life`) get downweighted without their genre-stock cousins (`sheriff`, `horse`) being hidden entirely.

In [14]:
freq['is_universal_strict'] = freq['n_genres_present'] >= UNIVERSAL_THRESHOLD
freq['breadth_penalty'] = freq['n_genres_present'] / N_GENRES

n_strict_universal = freq[freq['is_universal_strict']][['element', 'element_type']].drop_duplicates().shape[0]
print(f'Strict universals (appear in all 17 genres): {n_strict_universal:,} elements')

# Show what counts as strict-universal
strict_list = (freq[freq['is_universal_strict']]
               [['element', 'element_type', 'total_film_count']]
               .drop_duplicates()
               .sort_values('total_film_count', ascending=False))
print(f'\nTop 30 strict universals (by total film count):')
print(strict_list.head(30).to_string(index=False))

Strict universals (appear in all 17 genres): 5,500 elements

Top 30 strict universals (by total film count):
 element element_type  total_film_count
     man         noun             16261
    kill         verb             14699
    life         noun             14559
  friend         noun             14356
  return         verb             12521
    fall         verb             12484
    time         noun             12437
    love         noun             11610
  father         noun             11207
    live         verb             10570
discover         verb             10560
     run         verb             10264
  arrive         verb             10134
     end         verb              9816
   woman         noun              9631
    wife         noun              9523
   start         verb              9501
   house         noun              9381
  family         noun              9344
     die         verb              9147
  escape         verb              8932
  follow   

## 5. Primary Stockiness — Cognitive psychology grounding (Rosch's cue + category validity)

$$\text{PS}(e, g) = \underbrace{\frac{c_{eg}^{\text{film}}}{N_g}}_{\text{category validity: } P(e|g)} \times \underbrace{\frac{c_{eg}^{\text{film}}}{c_e^{\text{film}}}}_{\text{cue validity: } P(g|e)}$$

Rosch (1973, 1978): a feature's diagnosticity for a category is the product of cue validity (does seeing the feature tell you the category?) and category validity (does the category typically have the feature?). PS operationalizes both at the film level.

In [15]:
films_per_genre_series = pd.Series(films_per_genre)

freq['rel_freq_in_genre'] = freq['film_count'] / freq['genre'].map(films_per_genre_series)
freq['distinctiveness']   = freq['film_count'] / freq['total_film_count']
freq['primary_stockiness'] = freq['rel_freq_in_genre'] * freq['distinctiveness']

print(f'Primary Stockiness range: [{freq["primary_stockiness"].min():.5f}, {freq["primary_stockiness"].max():.5f}]')

Primary Stockiness range: [0.00000, 0.22107]


## 6. TF-IDF (Smoothed) — Information retrieval grounding

$$\text{TF-IDF}(e, g) = \frac{c_{eg}}{T_g} \times \left[ \log\frac{1 + 17}{1 + n_g(e)} + 1 \right]$$

**Why smoothed IDF, not vanilla.** With only 17 genres treated as "documents," vanilla IDF = log(17 / n_g(e)) becomes 0 for any element appearing in all 17 genres. That zeros out the TF-IDF of `sheriff`, `body`, `blood`, `death`, `house`, `ghost` — exactly the legally important universal-spanning stock elements. Smoothed IDF (sklearn / BM25 style) adds 1 to numerator and denominator and adds 1 to the result, ensuring every IDF is ≥ 1 while preserving the relative ordering (rarer-across-genres still scores higher).

For comparison:
- Sheriff (in 17/17 genres): vanilla IDF = log(17/17) = 0; smoothed IDF = log(18/18) + 1 = **1.0**
- Wagon train (in 7/17 genres): vanilla IDF ≈ 0.89; smoothed IDF = log(18/8) + 1 ≈ **2.81**
- Perfectly genre-specific element (1/17): vanilla IDF = log(17) ≈ 2.83; smoothed IDF = log(18/2) + 1 ≈ **3.20**


In [16]:
genre_total_series = pd.Series(genre_total_instances)
freq['tf'] = freq['instance_count'] / freq['genre'].map(genre_total_series)

# Smoothed IDF (sklearn / BM25 style): log((1 + N_g) / (1 + n_g(e))) + 1
# Prevents vanilla TF-IDF's degenerate behavior with few categorical "documents":
# vanilla IDF = log(17/17) = 0 zeros out every element that appears in all 17 genres,
# which destroys sheriff/body/blood/death/etc. — exactly the legally important stock elements.
freq['idf'] = np.log((1 + N_GENRES) / (1 + freq['n_genres_present'])) + 1
freq['tfidf'] = freq['tf'] * freq['idf']

print(f'TF-IDF (smoothed) range: [{freq["tfidf"].min():.6f}, {freq["tfidf"].max():.6f}]')
print(f'Elements with IDF = 1.0 (universal, in all 17 genres): {(freq["idf"] == 1.0).sum():,}')
print(f'  These now have non-zero TF-IDF (vanilla TF-IDF would zero them out).')


TF-IDF (smoothed) range: [0.000000, 0.012801]
Elements with IDF = 1.0 (universal, in all 17 genres): 93,500
  These now have non-zero TF-IDF (vanilla TF-IDF would zero them out).


## 7. PMI (kept, flagged fragile)

Standard pointwise mutual information in the multi-label-inflated sample space. We keep it for completeness and as a comparison point but exclude it from the ensemble because it's dominated by rare proper nouns. Use the film filter from section 3 to mitigate but not eliminate the problem.

In [17]:
N = freq['instance_count'].sum()
genre_marginals = freq.groupby('genre')['instance_count'].sum().to_dict()

freq['p_eg'] = freq['instance_count'] / N
freq['p_e']  = freq['total_instance_count'] / N
freq['p_g']  = freq['genre'].map(genre_marginals) / N
freq['pmi']  = np.log((freq['p_eg'] + 1e-12) / (freq['p_e'] * freq['p_g'] + 1e-12))
freq['pmi_positive'] = freq['pmi'].clip(lower=0)

print(f'PMI range: [{freq["pmi"].min():.4f}, {freq["pmi"].max():.4f}]')

PMI range: [-4.9426, 4.2598]


## 8. Dunning's G² — Corpus linguistics gold standard (Dunning 1993)

The keyness measure used by AntConc, Sketch Engine, and Lee & Mouritsen's legal-corpus work. Tests whether the observed frequency of element $e$ in genre $g$ is statistically more than expected under independence.

For 2×2 contingency table (element-in-g, element-in-other-genres, other-elements-in-g, other-elements-in-other):

$$G^2 = 2 \sum_i O_i \ln \frac{O_i}{E_i}$$

where $E_i$ are expected counts under independence. We compute the signed G² so positive values mean over-representation in genre $g$ (positive keyness). G² has chi-squared(1) asymptotic distribution, so values >3.84 are significant at p<0.05 and >10.83 are significant at p<0.001.

**Why this beats PMI:** PMI assigns extreme scores to rare events (a single film mentioning a proper noun 50 times). G² is asymptotically well-behaved and penalizes low-count cells appropriately. This is exactly why corpus linguistics uses G² as its primary keyword measure.

In [18]:
def compute_dunning_g2_vectorized(freq, N_total, genre_marginals):
    """Vectorized Dunning log-likelihood (G²) for every (element, genre) cell.

    a = freq of e in g                     (cell value: instance_count)
    b = freq of e in not-g                 (total_instance_count - a)
    c = total tokens in g                  (genre_marginals[g])
    d = total tokens in not-g              (N - c)

    Returns signed G² (positive = over-represented in g, negative = under-represented).
    """
    a = freq['instance_count'].values.astype(float)
    e_total = freq['total_instance_count'].values.astype(float)
    g_total = freq['genre'].map(genre_marginals).values.astype(float)

    b = e_total - a
    c = g_total - a   # other elements in this genre
    d = N_total - g_total - b  # other elements in other genres

    # Expected counts under independence (proportional)
    row1 = a + b  # total e
    row2 = c + d  # total not-e
    col1 = a + c  # total in g
    col2 = b + d  # total in not-g

    e_a = row1 * col1 / N_total
    e_b = row1 * col2 / N_total
    e_c = row2 * col1 / N_total
    e_d = row2 * col2 / N_total

    # Each term contributes O * log(O/E); guard log(0) by replacing 0*log(0) with 0
    def safe_term(o, e):
        with np.errstate(divide='ignore', invalid='ignore'):
            t = o * np.log(o / e)
        return np.where(o > 0, t, 0.0)

    g2 = 2 * (safe_term(a, e_a) + safe_term(b, e_b) + safe_term(c, e_c) + safe_term(d, e_d))

    # Sign: positive if element is over-represented in this genre vs. expected
    sign = np.where(a > e_a, 1.0, -1.0)

    return sign * g2

freq['dunning_g2'] = compute_dunning_g2_vectorized(freq, N, genre_marginals)
freq['dunning_g2_positive'] = freq['dunning_g2'].clip(lower=0)

print(f'Dunning G² range: [{freq["dunning_g2"].min():.2f}, {freq["dunning_g2"].max():.2f}]')
print(f'  Positive (over-represented): {(freq["dunning_g2"] > 0).sum():,} cells')
print(f'  Negative (under-represented): {(freq["dunning_g2"] < 0).sum():,} cells')
print(f'  Significant at p<0.001 (|G²| > 10.83): {(freq["dunning_g2"].abs() > 10.83).sum():,} cells')

Dunning G² range: [-4503.27, 7562.97]
  Positive (over-represented): 259,496 cells
  Negative (under-represented): 196,663 cells
  Significant at p<0.001 (|G²| > 10.83): 48,799 cells


## 9. Posterior Stockiness — Bayesian grounding

**Unsmoothed**

$$P(g \mid e) = \frac{c_{eg}^{\text{film}}}{c_e^{\text{film}}}$$

Of all films containing element $e$, what fraction are tagged with genre $g$? This is the directly legally-relevant quantity: when a court sees element $X$ in a defendant's work, what is the probability that work is genre $Y$?

The unsmoothed posterior is mathematically identical to the `distinctiveness` factor of Primary Stockiness ($c_{eg}^{\text{film}} / c_e^{\text{film}}$). We report it separately for transparency about that algebraic identity, but for the ensemble we use the **smoothed** version below so the Bayesian member of the ensemble is constructed independently from Primary.

**Dirichlet-smoothed (Section 9b)**

$$P_{\text{smoothed}}(g \mid e) = \frac{c_{eg}^{\text{film}} + \alpha\, P_{\text{prior}}(g)}{c_e^{\text{film}} + \alpha}$$

with $\alpha = 1$ (standard add-one) and $P_{\text{prior}}(g)$ = each genre's share of the corpus (film, genre) pairs. Smoothing toward the corpus-wide genre prior breaks the algebraic identity with Primary's distinctiveness factor, which is what we want for the ensemble.

In [19]:
freq['posterior_stockiness'] = freq['film_count'] / freq['total_film_count']

print(f'Posterior P(g|e) range: [{freq["posterior_stockiness"].min():.4f}, {freq["posterior_stockiness"].max():.4f}]')
print(f'(This equals distinctiveness; we keep both names for clarity in the paper.)')

Posterior P(g|e) range: [0.0009, 1.0000]
(This equals distinctiveness; we keep both names for clarity in the paper.)


## 9b. Dirichlet-Smoothed Posterior and Element-Genre Entropy

Two additions to the Bayesian and information-theoretic story.

**Smoothed posterior.** The unsmoothed posterior above is the distinctiveness factor of Primary Stockiness rewritten. To get a Bayesian member of the ensemble that is mathematically distinct from Primary, we add a Dirichlet smoothing toward the corpus-wide genre prior:

$$P_{\text{smoothed}}(g \mid e) = \frac{c_{eg}^{\text{film}} + \alpha\, P_{\text{prior}}(g)}{c_e^{\text{film}} + \alpha}, \quad \alpha = 1$$

The prior $P_{\text{prior}}(g)$ is each genre's share of the corpus (film, genre) pairs. With $\alpha = 1$ the smoothing pulls each element's genre distribution one prior-weighted observation toward the corpus baseline. Elements with many films are barely affected; elements near the filter threshold get a small, principled regularization. Sensitivity to $\alpha$ can be reported as a paper robustness check.

**Element-genre entropy.** Shannon entropy of the per-element genre distribution, using instance counts so the probabilities sum to 1:

$$H(G \mid e) = -\sum_g P(g \mid e)\, \log P(g \mid e), \quad P(g \mid e) = \frac{c_{eg}}{c_e}$$

Low entropy means the element concentrates in one or a few genres (high distinctiveness in the information-theoretic sense). Maximum entropy is $\log 17 \approx 2.833$, achieved when the element is spread evenly across all 17 genres. The normalized version $H/\log 17$ lives on $[0, 1]$ and is directly comparable to the existing `breadth_penalty`, which is the simple count $n_{\text{genres present}} / 17$. Entropy is the Shannon-grounded breadth measure weighted by mass; the count is not. Both are kept; neither enters the ensemble. (Dunning $G^2$, already in the ensemble, is asymptotically $2N$ times the KL divergence of the per-genre distribution against the prior, so the ensemble already has a Shannon-related member.)

In [20]:
ALPHA_SMOOTH = 1.0  # Dirichlet concentration parameter (standard add-one)

# Corpus-wide genre prior, film-based to match the units of the smoothed
# posterior. films_per_genre was computed in cell 8 from the multi-label
# explode; summing across genres gives the total (film, genre) pair count.
films_per_genre_total = int(films_per_genre.sum())
genre_prior_films = films_per_genre / films_per_genre_total
freq['p_prior_g'] = freq['genre'].map(genre_prior_films)

freq['posterior_smoothed'] = (
    (freq['film_count'] + ALPHA_SMOOTH * freq['p_prior_g'])
    / (freq['total_film_count'] + ALPHA_SMOOTH)
)

print(f'posterior_smoothed range: '
      f'[{freq["posterior_smoothed"].min():.4f}, {freq["posterior_smoothed"].max():.4f}]')
print(f'alpha = {ALPHA_SMOOTH}; corpus prior sums to {genre_prior_films.sum():.4f} '
      f'(should be 1.0)')

# Element-genre entropy. Computed from instance-based P(g|e), which sums to 1
# across genres by construction (instance_count / total_instance_count are
# both post-explode and consistent). The film-based P(g|e) above does NOT sum
# to 1 under multi-label genre tagging, so it would not give a proper entropy.
p_inst = freq['instance_count'] / freq['total_instance_count']
# guard log(0) although the filter prevents zero-count cells from reaching here
term = -p_inst * np.log(p_inst.clip(lower=1e-12))
freq['_entropy_term'] = term

entropy_per_elem = (freq.groupby(['element', 'element_type'])['_entropy_term']
                    .sum().reset_index(name='element_entropy'))
entropy_per_elem['element_entropy_normalized'] = (
    entropy_per_elem['element_entropy'] / np.log(N_GENRES)
)
freq = freq.merge(entropy_per_elem, on=['element', 'element_type'], how='left')
freq = freq.drop(columns=['_entropy_term'])

print(f'\nelement_entropy range:    '
      f'[{freq["element_entropy"].min():.4f}, {freq["element_entropy"].max():.4f}]')
print(f'  (max possible = log({N_GENRES}) = {np.log(N_GENRES):.4f})')
print(f'element_entropy_normalized range: '
      f'[{freq["element_entropy_normalized"].min():.4f}, '
      f'{freq["element_entropy_normalized"].max():.4f}]')

# Sanity: normalized entropy should rise monotonically (on average) with
# n_genres_present and asymptote near 1.0 at the strict-universal end.
print(f'\nSanity - normalized entropy by n_genres_present:')
sanity = (freq[['element', 'element_type', 'n_genres_present',
                'element_entropy_normalized']]
          .drop_duplicates(subset=['element', 'element_type'])
          .groupby('n_genres_present')['element_entropy_normalized']
          .agg(['mean', 'min', 'max', 'count']))
print(sanity.to_string())

posterior_smoothed range: [0.0009, 0.9863]
alpha = 1.0; corpus prior sums to 1.0000 (should be 1.0)

element_entropy range:    [0.0000, 2.6862]
  (max possible = log(17) = 2.8332)
element_entropy_normalized range: [0.0000, 0.9481]

Sanity - normalized entropy by n_genres_present:
                      mean       min       max  count
n_genres_present                                     
1                 0.000000  0.000000  0.000000      4
2                 0.217470  0.117391  0.244651     28
3                 0.342949  0.129764  0.387762    166
4                 0.419251  0.070274  0.489301    567
5                 0.497068  0.220548  0.568061   1367
6                 0.564641  0.291695  0.632412   2414
7                 0.617953  0.361908  0.684096   3417
8                 0.666063  0.366940  0.730095   3959
9                 0.702569  0.451794  0.768599   4016
10                0.733717  0.406200  0.812712   3657
11                0.759702  0.243007  0.837177   3352
12               

## 10. Ensemble — Robust Combined Score

Z-score each of {Primary Stockiness, TF-IDF, G²-positive, Posterior} within genre, then average. The four-measure ensemble excludes raw PMI because PMI's vulnerability to rare proper nouns made the v1 ensemble unreliable (it ranked single-film proper nouns above genuine genre conventions).

Within-genre z-scoring makes the measures comparable: each one is rescaled to ask 'how does this element rank under measure M *within this genre*', regardless of M's absolute scale.

In [21]:
def zscore_within_group(s):
    mu = s.mean()
    sd = s.std(ddof=0)
    if sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd

# Z-score all five quantities (the four ensemble measures plus the unsmoothed
# posterior for robustness). The ensemble uses posterior_smoothed_z, NOT
# posterior_stockiness_z: the unsmoothed posterior equals film_count /
# total_film_count, which is the distinctiveness factor inside Primary
# Stockiness, so including it in the ensemble would partially double-weight
# that factor. The Dirichlet-smoothed version mixes in the corpus genre
# prior, which breaks the algebraic identity and gives us a Bayesian member
# that is genuinely distinct from Primary.
for measure in ['primary_stockiness', 'tfidf', 'dunning_g2_positive',
                'posterior_smoothed', 'posterior_stockiness']:
    freq[f'{measure}_z'] = freq.groupby('genre')[measure].transform(zscore_within_group)

freq['ensemble'] = (
    freq['primary_stockiness_z']
    + freq['tfidf_z']
    + freq['dunning_g2_positive_z']
    + freq['posterior_smoothed_z']
) / 4

print(f'Ensemble range: [{freq["ensemble"].min():.3f}, {freq["ensemble"].max():.3f}]')
print(f'Ensemble mean (should be near 0 per genre): '
      f'{freq.groupby("genre")["ensemble"].mean().abs().max():.6f}')

# Robustness check: correlation between smoothed and unsmoothed posterior z-scores
corr = freq[['posterior_smoothed_z', 'posterior_stockiness_z']].corr().iloc[0, 1]
print(f'corr(posterior_smoothed_z, posterior_stockiness_z): {corr:.4f}')
print('  (high correlation expected; the smoothing only matters in the tails '
      'where total_film_count is small)')

Ensemble range: [-0.769, 67.151]
Ensemble mean (should be near 0 per genre): 0.000000
corr(posterior_smoothed_z, posterior_stockiness_z): 0.9967
  (high correlation expected; the smoothing only matters in the tails where total_film_count is small)


## 11. Sanity Check: Top K per Genre, Multiple Measures

This is the diagnostic that tells us whether the math actually works. We show the top 15 nouns under three measures (PS, G², Ensemble) for every genre. We **do not** exclude universals here — universals get downweighted by their breadth penalty but stay visible so we can see whether `sheriff` appears at the top for Western (which would mean the math works) or only `cowhand` (which would mean the universal-hiding bug from v1 is still happening).

**What to look for:**
- Western should show sheriff, horse, gun, town, gang, ranch, outlaw, cowboy
- Horror should show blood, body, killer, ghost, vampire, monster, demon, zombie
- Sci-Fi should show alien, planet, spaceship, robot, earth, space, world
- Romance should show love, wedding, kiss, marriage, heart, affair
- War should show soldier, army, battle, mission, officer, war, camp
- Crime should show police, murder, gun, money, gang, detective

In [22]:
def top_k_for_genre(df, genre, measure, k=15, types=('noun',)):
    """Return top K elements for a genre under a given measure. Universals stay visible.

    Returns columns: element, element_type, <measure>, film_count,
    n_genres_present, is_universal_strict. The element_type column is
    included so downstream code (per-type top-K saves, reports that mix
    types) can render labels and trace which element_type produced a row.
    """
    g = df[df['genre'] == genre]
    if types:
        g = g[g['element_type'].isin(types)]
    return g.nlargest(k, measure)[['element', 'element_type', measure,
                                    'film_count', 'n_genres_present',
                                    'is_universal_strict']]

print('=' * 100)
print(f'{"TOP 15 STOCK NOUNS PER GENRE (Primary Stockiness, universals VISIBLE)":^100}')
print('=' * 100)

for genre in CANONICAL_GENRES:
    top = top_k_for_genre(freq, genre, 'primary_stockiness', k=15)
    print(f'\n{genre.upper()}')
    for i, (_, row) in enumerate(top.iterrows(), 1):
        u = '*' if row['is_universal_strict'] else ' '
        print(f'  {i:2d}.{u} {row["element"]:<22} PS={row["primary_stockiness"]:.4f}  '
              f'films={int(row["film_count"]):>4}  in {int(row["n_genres_present"]):>2}/17 genres')
print('\n(* = appears in all 17 genres)')

               TOP 15 STOCK NOUNS PER GENRE (Primary Stockiness, universals VISIBLE)                

ACTION
   1.* man                    PS=0.0931  films=3430  in 17/17 genres
   2.* death                  PS=0.0639  films=2083  in 17/17 genres
   3.* battle                 PS=0.0575  films=1049  in 17/17 genres
   4.* officer                PS=0.0560  films=1281  in 17/17 genres
   5.* group                  PS=0.0551  films=1553  in 17/17 genres
   6.* fight                  PS=0.0548  films=1288  in 17/17 genres
   7.* force                  PS=0.0508  films=1059  in 17/17 genres
   8.* friend                 PS=0.0503  films=2369  in 17/17 genres
   9.* order                  PS=0.0499  films=1544  in 17/17 genres
  10.* life                   PS=0.0497  films=2373  in 17/17 genres
  11.* mission                PS=0.0493  films= 878  in 17/17 genres
  12.* gang                   PS=0.0492  films=1037  in 17/17 genres
  13.* attack                 PS=0.0487  films=1142  in 17/17 g

In [23]:
# Side-by-side comparison: PS vs Dunning G² vs Ensemble for illustrative genres
ILLUSTRATIVE_GENRES = ['Western', 'Horror', 'Science Fiction', 'Romance', 'War', 'Crime']

for genre in ILLUSTRATIVE_GENRES:
    print(f'\n{"="*100}')
    print(f'  {genre} — top 10 nouns under each robust measure')
    print('=' * 100)

    ps  = top_k_for_genre(freq, genre, 'primary_stockiness', k=10)['element'].tolist()
    g2  = top_k_for_genre(freq, genre, 'dunning_g2_positive', k=10)['element'].tolist()
    pst = top_k_for_genre(freq, genre, 'posterior_stockiness', k=10)['element'].tolist()
    ens = top_k_for_genre(freq, genre, 'ensemble', k=10)['element'].tolist()

    print(f'  {"Rank":<6}{"Primary Stockiness":<22}{"Dunning G²":<22}{"Posterior P(g|e)":<22}{"Ensemble":<22}')
    for i in range(10):
        row = [ps, g2, pst, ens]
        cells = [(r[i] if i < len(r) else '') for r in row]
        print(f'  {i+1:<6}{cells[0]:<22}{cells[1]:<22}{cells[2]:<22}{cells[3]:<22}')


  Western — top 10 nouns under each robust measure
  Rank  Primary Stockiness    Dunning G²            Posterior P(g|e)      Ensemble              
  1     ranch                 ranch                 blackfeet             ranch                 
  2     outlaw                town                  carpetbagger          town                  
  3     saloon                horse                 custer                horse                 
  4     posse                 indian                earp                  outlaw                
  5     sheriff               sheriff               saddlebag             man                   
  6     stagecoach            outlaw                sioux                 sheriff               
  7     indian                posse                 apach                 posse                 
  8     cattle                saloon                gunfighter            indian                
  9     horse                 cattle                rustler               s

In [24]:
# Sanity check: where does 'sheriff' rank for Western under each measure?
print('=== Diagnostic: position of "sheriff" in Western under each measure ===\n')

for measure in ['primary_stockiness', 'tfidf', 'dunning_g2_positive', 'posterior_stockiness', 'ensemble']:
    western = (freq[(freq['genre']=='Western') & (freq['element_type']=='noun')]
               .sort_values(measure, ascending=False)
               .reset_index(drop=True))
    sheriff_idx = western[western['element'] == 'sheriff'].index
    if len(sheriff_idx) > 0:
        rank = sheriff_idx[0] + 1
        score = western.iloc[sheriff_idx[0]][measure]
        print(f'  {measure:<25} sheriff rank: #{rank:>4}  score: {score:.4f}')
    else:
        print(f'  {measure:<25} sheriff not found in Western (filtered out?)')

=== Diagnostic: position of "sheriff" in Western under each measure ===

  primary_stockiness        sheriff rank: #   5  score: 0.0648
  tfidf                     sheriff rank: #  12  score: 0.0029
  dunning_g2_positive       sheriff rank: #   5  score: 1463.2063
  posterior_stockiness      sheriff rank: # 140  score: 0.3558
  ensemble                  sheriff rank: #   6  score: 14.6463


## 12. Save Outputs

In [ ]:
out_cols = [
    'element', 'element_type', 'genre',
    'instance_count', 'film_count',
    'total_instance_count', 'raw_instance_count', 'total_film_count', 'n_genres_present',
    'rel_freq_in_genre', 'distinctiveness',
    'primary_stockiness',
    'tf', 'idf', 'tfidf',
    'pmi', 'pmi_positive',
    'dunning_g2', 'dunning_g2_positive',
    'posterior_stockiness',     # unsmoothed (= distinctiveness; retained for transparency)
    'posterior_smoothed',       # Dirichlet-smoothed; the Bayesian ensemble member
    'p_prior_g',                # corpus-wide genre prior used in smoothing
    'element_entropy',          # Shannon entropy of P(g|e) [per element]
    'element_entropy_normalized',  # entropy / log(N_GENRES), on [0, 1]
    'primary_stockiness_z', 'tfidf_z', 'dunning_g2_positive_z',
    'posterior_smoothed_z',     # the ensemble's Bayesian z-component
    'posterior_stockiness_z',   # unsmoothed Bayesian z, for robustness checks
    'ensemble',
    'is_universal_strict', 'breadth_penalty'
]

out_long = freq[out_cols].copy()
long_path = f'{OUTPUT_DIR}/stockiness_long.parquet'
out_long.to_parquet(long_path, engine='pyarrow', compression='snappy')
print(f'Saved: {long_path}  ({os.path.getsize(long_path)/1e6:.1f} MB, {len(out_long):,} rows)')

# Top-K per (genre, measure), broken out by element_type group. Five files:
#   - nouns    : single-word common nouns
#   - chunks   : multi-word noun phrases (e.g. "wagon train", "magic mirror")
#   - verbs    : action verbs treated as plot beats (rescue, betray, haunt)
#   - entities : named entities (entity_GPE, entity_ORG, entity_NORP, ...)
#   - overall  : all element types combined, single ranking per (genre, measure)
# Each file has identical schema and covers every measure including the
# ensemble, so the doctrinal picture (Primary, Dunning, Posterior_smoothed,
# Ensemble + the other diagnostics) is visible for each element_type slice
# and for the combined ranking. The overall file is dominated by nouns
# because their per-cell counts are larger, but the per-type files give
# each element_type its own ranked top-50.
TOP_FOR_PAPER = 50
TYPE_GROUPS = [
    ('nouns',    ('noun',)),
    ('chunks',   ('chunk',)),
    ('verbs',    ('verb',)),
    ('entities', ('entity_ORG', 'entity_GPE', 'entity_NORP', 'entity_LOC',
                  'entity_PRODUCT', 'entity_FAC', 'entity_WORK_OF_ART',
                  'entity_EVENT')),
    ('overall',  None),
]
MEASURES_FOR_TOPK = [
    'primary_stockiness', 'tfidf', 'dunning_g2_positive',
    'posterior_smoothed', 'posterior_stockiness',
    'ensemble', 'pmi_positive',
]

for group_name, types in TYPE_GROUPS:
    rows = []
    for genre in CANONICAL_GENRES:
        for measure in MEASURES_FOR_TOPK:
            top = top_k_for_genre(freq, genre, measure,
                                   k=TOP_FOR_PAPER, types=types)
            for rank, (_, row) in enumerate(top.iterrows(), 1):
                rows.append({
                    'genre': genre,
                    'measure': measure,
                    'rank': rank,
                    'element': row['element'],
                    'element_type': row['element_type'],
                    'score': row[measure],
                    'film_count': row['film_count'],
                    'n_genres_present': row['n_genres_present'],
                    'is_universal_strict': row['is_universal_strict'],
                })
    top_df = pd.DataFrame(rows)
    out_path = f'{OUTPUT_DIR}/top_stock_by_genre_{group_name}.parquet'
    top_df.to_parquet(out_path, engine='pyarrow', compression='snappy')
    print(f'Saved: {out_path}  '
          f'({os.path.getsize(out_path)/1e6:.2f} MB, {len(top_df):,} rows)')

# Remove the legacy combined file if it exists from a prior run. The five
# per-type files supersede it and the schema has changed (element_type
# column added), so keeping the old file around would only confuse readers.
legacy_path = f'{OUTPUT_DIR}/top_stock_by_genre.parquet'
if os.path.exists(legacy_path):
    os.remove(legacy_path)
    print(f'Removed legacy: {legacy_path}')

In [ ]:
report_path = f'{OUTPUT_DIR}/stockiness_report.txt'

# Element-type groups mirror the per-type parquet outputs in the previous
# cell. Each group gets its own report section with top-10 per genre under
# four measures (Primary, Dunning, smoothed Posterior, Ensemble). The
# "overall" group prints element_type alongside each element so a chunk
# topping a list reads as "wagon train [chunk]" rather than being
# indistinguishable from a noun.
REPORT_TYPE_GROUPS = [
    ('OVERALL',  None,                                                       'all element types combined'),
    ('NOUNS',    ('noun',),                                                  'single-word common nouns'),
    ('CHUNKS',   ('chunk',),                                                 'multi-word noun phrases'),
    ('VERBS',    ('verb',),                                                  'action verbs'),
    ('ENTITIES', ('entity_ORG', 'entity_GPE', 'entity_NORP', 'entity_LOC',
                  'entity_PRODUCT', 'entity_FAC', 'entity_WORK_OF_ART',
                  'entity_EVENT'),                                            'named entities'),
]
REPORT_MEASURES = [
    ('primary_stockiness',  'Primary Stockiness',         'PS',  '.4f'),
    ('dunning_g2_positive', 'Dunning G^2',                'G2',  '.1f'),
    ('posterior_smoothed',  'Posterior P(g|e) smoothed',  'P_s', '.4f'),
    ('ensemble',            'Ensemble (z-avg of 4)',      'ens', '.3f'),
]

with open(report_path, 'w') as f:
    f.write('STOCKINESS COMPUTATION REPORT (Revised)\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Element index rows:        {len(ei):,}\n')
    f.write(f'After explode:             {len(joined):,}\n')
    f.write(f'Frequency threshold:       at least {MIN_FREQ_THRESHOLD} instances (each instance counted once per film)\n')
    f.write(f'Film threshold:            at least {MIN_FILMS_PER_ELEMENT} films\n')
    f.write(f'Cells after filter:        {len(freq):,}\n')
    f.write(f'Unique elements kept:      {n_unique_elements_kept:,}\n')
    f.write(f'  of which strict-universal: {n_strict_universal:,}\n')
    f.write(f'Universality threshold:    {UNIVERSAL_THRESHOLD}/17 (binary tag only)\n')
    f.write(f'Dirichlet smoothing alpha: {ALPHA_SMOOTH}\n\n')

    f.write('MEASURES INCLUDED\n')
    for m in ['Primary Stockiness (Rosch cue validity)',
              'TF-IDF (smoothed)',
              "Dunning G^2 (corpus linguistics; asymptotic KL divergence)",
              'Posterior P(g|e), Dirichlet-smoothed (Bayesian, ensemble member)',
              'Ensemble (z-avg of the four above)']:
        f.write(f'  - {m}\n')
    f.write('  - Posterior P(g|e), unsmoothed (kept for transparency: identical to distinctiveness)\n')
    f.write('  - Element-genre entropy H(G|e) (Shannon breadth measure, not in ensemble)\n')
    f.write('  - PMI (kept for completeness, excluded from ensemble)\n\n')

    f.write('VOCABULARY COUNTS PER ELEMENT_TYPE (cells after the 10-instance / 5-film filter)\n')
    type_counts = freq.groupby('element_type').size().sort_values(ascending=False)
    for et, n in type_counts.items():
        f.write(f'  {et:<24} {int(n):>8,}\n')
    f.write('\n')

    # Per-type sections, each with top-10 per genre under four measures.
    for group_name, types, descr in REPORT_TYPE_GROUPS:
        f.write('=' * 70 + '\n')
        f.write(f'TOP 10 PER GENRE - {group_name} ({descr})\n')
        f.write('=' * 70 + '\n\n')
        for measure_col, measure_label, score_abbrev, score_fmt in REPORT_MEASURES:
            f.write(f'--- {measure_label} ---\n\n')
            for g in CANONICAL_GENRES:
                f.write(f'{g}:\n')
                top = top_k_for_genre(freq, g, measure_col, k=10, types=types)
                if len(top) == 0:
                    f.write('  (no elements of this type in this genre after filter)\n\n')
                    continue
                for i, (_, row) in enumerate(top.iterrows(), 1):
                    et_tag = f' [{row["element_type"]}]' if group_name == 'OVERALL' else ''
                    score = format(row[measure_col], score_fmt)
                    f.write(f'  {i:2d}. {row["element"]:<28}{et_tag:<22} '
                            f'{score_abbrev}={score:>10}  '
                            f'(in {int(row["n_genres_present"])}/17 genres)\n')
                f.write('\n')
            f.write('\n')

    # Entropy section (unchanged).
    f.write('=' * 70 + '\n')
    f.write('LOWEST AND HIGHEST ELEMENT-GENRE ENTROPY\n')
    f.write('(low entropy = concentrated in one genre; high = spread evenly)\n')
    f.write('=' * 70 + '\n\n')
    elem_view = (freq[['element', 'element_type', 'n_genres_present',
                       'element_entropy', 'element_entropy_normalized']]
                 .drop_duplicates(subset=['element', 'element_type']))
    f.write('  Most concentrated (lowest entropy, among elements in 3+ genres):\n')
    low = (elem_view[elem_view['n_genres_present'] >= 3]
           .nsmallest(15, 'element_entropy_normalized'))
    for _, row in low.iterrows():
        f.write(f'    {row["element"]:<28} [{row["element_type"]:<18}] '
                f'H_norm={row["element_entropy_normalized"]:.3f}'
                f'  (in {int(row["n_genres_present"])}/17 genres)\n')
    f.write('\n  Most spread (highest entropy):\n')
    high = elem_view.nlargest(15, 'element_entropy_normalized')
    for _, row in high.iterrows():
        f.write(f'    {row["element"]:<28} [{row["element_type"]:<18}] '
                f'H_norm={row["element_entropy_normalized"]:.3f}'
                f'  (in {int(row["n_genres_present"])}/17 genres)\n')

print(f'Saved report: {report_path}')

## 13. Decisions for Notebook 04

**Q1: Does `sheriff` now appear at the top for Western?** Check the diagnostic in section 11. If yes, the universality fix worked. If no, we need to investigate further.

**Q2: Do PS and Dunning G² agree on the top elements per genre?** Section 11's side-by-side comparison. If they agree closely (rank correlation > 0.7 by eye), we have multi-measure convergence and the signal is robust. If they disagree dramatically, that's worth diagnosing.

**Q3: Does the Ensemble look reasonable now?** Without PMI's hapax problem polluting it, the ensemble should produce stable top elements that overlap heavily with PS and G². No more `rocklin` at rank 1.

**Future directions (not for v1):**
- **Network centrality** within each genre's element co-occurrence network. Eigenvector centrality identifies elements that are *constitutive* of the genre, mapping onto §18's 'indispensable' language. This deserves its own Notebook 3b.
- **Multi-word element extraction** in Notebook 2. Current pipeline collapses 'haunted house' → 'house' via lemma-root extraction. The Sonnet validation in 02b confirmed this is the largest extraction gap. A future revision should extract full noun chunks alongside single-word lemmas.
- **Character-name post-filter** using Wikipedia/Wikidata lookup to catch the proper nouns spaCy's NER missed (Daffy, Shekhar, Moriarty, etc.).

**Next: Notebook 04** loads the union corpus and computes plot-level surprisal on the full 45,958 films using a modern long-context LM (Qwen2.5-7B or Llama-3.1-8B) on the A100. Output: per-film surprisal scores that Notebook 06 will correlate against per-film mean stockiness to produce the headline genre-specific correlation findings.